# 08 — The NIN / PIP Isotype Diode
## From Ohm's law to space-charge-limited current: when a "resistor" stops being ohmic

An **isotype** structure puts a lightly doped layer between two heavily doped contacts of the *same* type: n⁺/n⁻/n⁺ (NIN)
or p⁺/p⁻/p⁺ (PIP). There is no p-n junction, so the device does not rectify. Yet it is not a simple resistor either: the
contacts inject carriers into the lightly doped layer, and at high bias the current is limited by those carriers' own
space charge. This is the physics of drift regions, detectors and organic semiconductors.

**Learning objectives**
1. Show from symmetry that an isotype device cannot rectify, and verify it.
2. Predict the low-bias resistance from Ohm's law, then see why it fails when the Debye length is comparable to the layer.
3. Derive the Mott–Gurney space-charge-limited current $J\propto V^2/L^3$ and compare with simulation.
4. Use the log-log slope $d\ln I/d\ln V$ to identify conduction regimes.
5. Compare electrons (NIN) with holes (PIP).

**Device** (factory default): 2 µm long; n⁺ contacts 10¹⁸ cm⁻³; a 0.8 µm n⁻ layer doped 10¹⁴ cm⁻³ starting at 0.6 µm.

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────
# PADRE is a compiled simulator. On nanoHUB it lives in an environment
# module that must be loaded into this Python kernel before the first
# simulation runs; elsewhere, PADRE just needs to be on your PATH.
import shutil
from nanohubpadre import use

if shutil.which("padre") is None:
    try:
        use("padre-2.4E-r15")          # nanoHUB module name
    except Exception as exc:           # not on nanoHUB: that is fine
        print("Could not load the PADRE module:", exc)

print("PADRE executable:", shutil.which("padre") or "NOT FOUND - simulations will fail")

In [ ]:
# ── Physical constants: textbook values vs. the values PADRE really uses ──
# PADRE prints its constants with `MODELS ... PRINT`. A fair comparison of
# simulation against a formula must feed the formula PADRE's numbers:
# e.g. the textbook n_i = 1.5e10 cm^-3 alone changes every analytic diode
# current by (1.5e10 / 9.963e9)^2 = 2.3x before any physics is compared.
import numpy as np

Q      = 1.602e-19        # elementary charge (C)
KT     = 0.025851         # thermal voltage kT/q at 300 K (V)
EPS0   = 8.854e-14        # vacuum permittivity (F/cm)
EPS_SI = 11.8 * EPS0      # silicon permittivity in PADRE (F/cm)
EPS_OX = 3.9 * EPS0       # SiO2 permittivity (F/cm)
NI     = 9.963e9          # intrinsic carrier density of Si at 300 K (cm^-3)
EG     = 1.12             # band gap (eV)
CHI_SI = 4.17             # electron affinity of Si in PADRE (eV)
NC, NV = 3.2e19, 2.03e19  # effective densities of states (cm^-3)

# The values most textbooks quote (e.g. Sze & Ng, Pierret), for contrast.
TEXTBOOK = dict(NI=1.5e10, EPS_SI=11.7 * EPS0, CHI_SI=4.05, NC=2.8e19, NV=1.04e19)


def compare(rows, title="Textbook vs. PADRE"):
    """Print a comparison table.

    rows: list of (quantity, textbook_value, padre_value, unit, why_different)
    """
    print(title)
    print("-" * 125)
    print(f"{'quantity':<40}{'textbook':>12}{'PADRE':>12}{'diff':>9}   why they differ")
    print("-" * 125)
    for name, tb, pd, unit, why in rows:
        d = f"{(pd - tb) / tb * 100:>+8.1f}%" if tb else "     n/a "
        print(f"{name + ' (' + unit + ')':<40}{tb:>12.4g}{pd:>12.4g}{d}   {why}")
    print("-" * 125)


compare([
    ("intrinsic density n_i", TEXTBOOK["NI"], NI, "cm^-3",
     "modern value (Sproul & Green 1991); 1.5e10 is the older textbook number"),
    ("Si permittivity", TEXTBOOK["EPS_SI"], EPS_SI, "F/cm", "PADRE uses eps_r = 11.8, Sze uses 11.7"),
    ("electron affinity chi", TEXTBOOK["CHI_SI"], CHI_SI, "eV", "PADRE (like Silvaco Atlas) uses 4.17"),
    ("conduction-band DOS Nc", TEXTBOOK["NC"], NC, "cm^-3", "different effective-mass fit"),
    ("valence-band DOS Nv", TEXTBOOK["NV"], NV, "cm^-3", "different effective-mass fit"),
], title="Material constants")

---
## 1. Theory

**Ohmic regime.** If the n⁻ layer held only its own electrons ($n=N_D$), it would be a resistor:
$$R=\frac{L}{qN_D\mu_nA},\qquad I=V/R.$$

**Space-charge-limited current (Mott & Gurney, 1940).** When the contact injects more carriers than the layer's own
doping, the injected charge controls the field. Combining $J=q\mu nE$ (drift only) with Poisson's equation
$dE/dx=qn/\varepsilon_s$ (only injected charge), and requiring $E=0$ at the injecting contact:
$$J_{SCL}=\frac98\,\varepsilon_s\mu\,\frac{V^2}{L^3}.$$
The crossover from Ohm's law occurs near $V_\times\approx\frac89\,qN_DL^2/\varepsilon_s$.

**Assumptions in Mott–Gurney:** no diffusion current, no background doping, constant mobility, and a perfect injecting
contact with $E=0$ there. We will check each one.

**A length scale to watch:** the Debye length $L_D=\sqrt{\varepsilon_skT/q^2N_D}$ is how far carriers from the n⁺
contacts spill into the n⁻ layer at zero bias.

In [ ]:
L_B = 0.8e-4                     # barrier (n-) width, cm
N_B = 1e14
MU_N, MU_P = 1382.0, 467.6       # PADRE conmob at 1e14 (notebook 09)
AREA = 1e-8                      # 1 um x 1 um
LD = np.sqrt(EPS_SI * KT / (Q * N_B))
R = L_B / (Q * N_B * MU_N * AREA)
V_x = 8 / 9 * Q * N_B * L_B ** 2 / EPS_SI
print(f"Debye length at 1e14: {LD * 1e4:.2f} um  (layer is {L_B * 1e4:.1f} um: comparable!)")
print(f"ohmic resistance of the n- layer: {R:.3e} ohm um")
print(f"Ohm -> Mott-Gurney crossover voltage: {V_x * 1e3:.0f} mV")

def i_ohm(V, mu=MU_N):
    return Q * N_B * mu * AREA * np.asarray(V) / L_B

def i_mg(V, mu=MU_N):
    return 9 / 8 * EPS_SI * mu * np.asarray(V) ** 2 / L_B ** 3 * AREA

---
## 2. Symmetry: an isotype device cannot rectify
Swapping the two contacts leaves the structure unchanged, so $I(-V)=-I(V)$ exactly. We sweep from −2 V to +2 V.

In [ ]:
from nanohubpadre import create_nin_diode

sim = create_nin_diode(log_iv=True, bias_sweep=(-2.0, 2.0, 0.1))
print(sim.run().returncode)
iv = sim.get_iv_data()
V, I = iv.get_voltages(1), iv.get_currents(1)
o = np.argsort(V); V, I = V[o], I[o]
_, u = np.unique(V, return_index=True); V, I = V[u], I[u]
pos = V > 0.05
asym = [abs(abs(np.interp(-v, V, I)) - abs(i)) / abs(i) for v, i in zip(V[pos], I[pos])]
print(f"max |I(+V)| vs |I(-V)| mismatch: {max(asym) * 100:.4f} %")
err = iv.continuity_error()
print(f"max terminal-current continuity error (|V| > 0.05 V): {err[np.abs(iv.get_voltages(1)) > 0.05].max():.1e}")

The I–V is antisymmetric to better than 0.01 %. The simulated structure is symmetric, and so is the physics.
This is a useful sanity check for any simulator: a symmetric device that "rectifies" has a bug.

---
## 3. Regimes of conduction

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

p = V > 0.02
Vp, Ip = V[p], np.abs(I[p])
slope = np.gradient(np.log(Ip), np.log(Vp))

fig = make_subplots(rows=1, cols=2, subplot_titles=("I-V (log-log)", "log-log slope d ln I / d ln V"))
fig.add_trace(go.Scatter(x=Vp, y=Ip, name="PADRE NIN", mode="markers+lines"), 1, 1)
fig.add_trace(go.Scatter(x=Vp, y=i_ohm(Vp), name="Ohm (n = N_D)", line=dict(dash="dot")), 1, 1)
fig.add_trace(go.Scatter(x=Vp, y=i_mg(Vp), name="Mott-Gurney", line=dict(dash="dash")), 1, 1)
fig.add_trace(go.Scatter(x=Vp, y=slope, name="slope", showlegend=False), 1, 2)
for y0, lab in ((1, "ohmic"), (2, "SCLC")):
    fig.add_hline(y=y0, line_dash="dot", annotation_text=lab, row=1, col=2)
fig.update_xaxes(type="log", title_text="V (V)")
fig.update_yaxes(type="log", title_text="A/um", row=1, col=1)
fig.update_layout(template="plotly_white", width=1050, height=450)
fig.show()

rows = []
for v in (0.1, 0.5, 2.0):
    k = np.argmin(abs(Vp - v))
    rows.append((f"I at {Vp[k]:.1f} V vs Ohm", float(i_ohm(Vp[k])), Ip[k], "A/um", "n in the layer >> N_D (contact spill-over)"))
    rows.append((f"I at {Vp[k]:.1f} V vs Mott-Gurney", float(i_mg(Vp[k])), Ip[k], "A/um", "diffusion, background doping, fldmob"))
compare(rows, title="NIN current: Ohm's law and Mott-Gurney vs PADRE")

**What is different, and why**
* **Ohm's law fails even at low bias.** The simple resistor model assumes the n⁻ layer holds $N_D$ = 10¹⁴ electrons per cm³.
  But the Debye length at 10¹⁴ is 0.41 µm, half the layer's width, so electrons from the n⁺ contacts diffuse deep into it
  at zero bias. The layer is far more conductive than its doping suggests.
* **Mott–Gurney overestimates at high bias.** Its derivation neglects diffusion, which opposes the drift of injected
  carriers near the contacts, and assumes constant mobility. At 2 V over 0.8 µm ($E\sim2.5\times10^4$ V/cm), field-dependent
  mobility already reduces the electron velocity.
* **The log-log slope** tells the story without any fitting: it rises from ~1 toward 2 as injection takes over, then sags
  as velocity saturation sets in. A pure velocity-saturated space-charge current would scale as $V/L^2$ (slope 1).

---
## 4. Electrons vs. holes: NIN vs. PIP

In [ ]:
sim_p = create_nin_diode(device_type="pip", log_iv=True, bias_sweep=(0.0, 2.0, 0.1))
print(sim_p.run().returncode)
ivp = sim_p.get_iv_data()
Vq, Iq = ivp.get_voltages(1), np.abs(ivp.get_currents(1))
kq, kn = np.argmax(Vq), np.argmin(abs(V - 2.0))
compare([
    ("I_NIN / I_PIP at 2 V", MU_N / MU_P, abs(I[kn]) / Iq[kq], "-",
     "electrons are closer to velocity saturation, lowering the NIN current"),
], title="Electrons vs holes")

Both SCLC and ohmic currents are proportional to $\mu$, so the textbook expects the current ratio to equal the mobility
ratio, 2.96. PADRE gives ~2.3, about 22 % lower. At ~2.5×10⁴ V/cm electrons, with their higher low-field mobility, are
much closer to their saturation velocity than holes are, so field-dependent mobility cuts the NIN current more
than the PIP current. Notebook 09 shows the two velocity-field curves.

---
## 5. Inside the barrier

In [ ]:
s = create_nin_diode(log_physics_at=[0.0, 0.5, 1.0, 2.0])
print(s.run().returncode)
fig = go.Figure()
for tag, lab in (("eq", "0 V"), ("v0p5", "0.5 V"), ("v1p0", "1.0 V"), ("v2p0", "2.0 V")):
    d = s.outputs.get(f"n_{tag}")
    fig.add_trace(go.Scatter(x=d.x, y=d.y, name=lab))
fig.add_hline(y=N_B, line_dash="dot", annotation_text="background doping 1e14")
fig.add_vrect(x0=0.6, x1=1.4, fillcolor="gray", opacity=0.1, annotation_text="n- layer")
fig.update_layout(yaxis_type="log", xaxis_title="position (um)", yaxis_title="electron density (cm^-3)",
                  template="plotly_white", width=800, height=450)
fig.show()

At 0 V the electron density in the n⁻ layer is already far above 10¹⁴ near both interfaces: that is the Debye spill-over.
Under bias the injecting (source) side fills further while the collecting side empties, which produces the asymmetric
space-charge profile Mott–Gurney describes.

---
## 6. Summary: textbook vs. simulation for the isotype diode

| Quantity | Textbook | PADRE | What differs |
|---|---|---|---|
| Symmetry | $I(-V)=-I(V)$ | exact to <0.01 % | — |
| Low-bias current | $V/R$ with $n=N_D$ | much larger | contact spill-over ($L_D\sim$ layer width) |
| High-bias current | Mott–Gurney $V^2/L^3$ | below it | diffusion, background doping, velocity saturation |
| NIN/PIP ratio | $\mu_n/\mu_p$ | close | different field-dependent mobility shapes |

---
## 7. Exercises
1. **Make it ohmic.** Raise `intrinsic_doping` to 1e16. $L_D$ drops to 0.04 µm. Is the low-bias current now close to $V/R$?
2. **Make it SCLC.** Lower it to 1e12. Where does the slope-2 region start? Compare with $V_\times$.
3. **Scaling with thickness.** Double `intrinsic_width`. Mott–Gurney predicts $I\propto L^{-3}$; Ohm predicts $L^{-1}$. Which do you see at 2 V?

**Next →** [09 — Validating a Simulator](09_Validating_a_Simulator.ipynb)